# Train and test tiny preselection embeddings

End-to-end workflow for the first-stage dedupe model: train a sub-1M-parameter character CNN, embed names/strings, and inspect the top 100 candidates that should be passed to a stricter verifier.


The default model is intentionally small: a token-aware character CNN with order-invariant token pooling, order-sensitive token pooling, and a whole-string branch. On the current defaults it is roughly 200k parameters, well below the 1M-parameter ceiling, so it should be fast enough for broad candidate preselection.

Training scores triplet candidates just in time instead of building a full all-pairs similarity matrix, so setup stays manageable as the corpus grows. For real dedupe work, replace the bundled words with representative names from your database. The bundled word list keeps this notebook runnable anywhere.


In [10]:
import importlib
import sys

sys.modules.pop("model", None)
sys.modules.pop("train", None)
import model as model_module
import train as train_module

model_module = importlib.reload(model_module)
train_module = importlib.reload(train_module)
print("model reloaded from", model_module.__file__)
print("has count_parameters", hasattr(model_module, "count_parameters"))


model reloaded from /workspaces/string-embed/model.py
has count_parameters True


In [11]:
from importlib import reload
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import random
import time
import torch

import model as model_module
import train as train_module
from testdata import load_words

reload(model_module)
reload(train_module)

DEFAULT_CANDIDATE_COUNT = model_module.DEFAULT_CANDIDATE_COUNT
MODEL_PARAMETER_LIMIT = model_module.MODEL_PARAMETER_LIMIT
StringEmbedder = model_module.StringEmbedder
count_parameters = model_module.count_parameters

name_similarity = train_module.name_similarity
nearest = train_module.nearest
train = train_module.train

device = "cuda" if torch.cuda.is_available() else "cpu"
context_chars = 32
candidate_count = DEFAULT_CANDIDATE_COUNT
sample_size = 64
lr = 1e-3
word_limit = 10_000
seed = 13
output_path = Path("tiny_string_embed.pt")
resume_from = None

if True:
    resume_from = output_path

words = load_words(limit=word_limit, seed=seed)
probe_model = StringEmbedder(max_len=context_chars)
model_info = {
    "model": "StringEmbedder token-aware char-CNN",
    "parameters": count_parameters(probe_model),
    "parameter_limit": MODEL_PARAMETER_LIMIT,
    "candidate_count": candidate_count,
    "training_sample_size": sample_size,
    "lr": lr,
    "device": device,
    "word_limit": word_limit,
    "seed": seed,
    "word_count": len(words),
    "resume_from": str(resume_from) if resume_from else None,
}
model_info, words[:10]


({'model': 'StringEmbedder token-aware char-CNN',
  'parameters': 200544,
  'parameter_limit': 1000000,
  'candidate_count': 100,
  'training_sample_size': 64,
  'lr': 0.001,
  'device': 'cuda',
  'word_limit': 10000,
  'seed': 13,
  'word_count': 10000,
  'resume_from': 'tiny_string_embed.pt'},
 ['abandonable',
  'abandoners',
  'abannition',
  'abasgi',
  'abbas',
  'abbess',
  'abdominogenital',
  'abdominothoracic',
  'abetment',
  'abietineae'])

In [12]:
result = train(
    words,
    epochs=50,
    batch_size=8192,
    lr=lr,
    progress=True,
    device=device,
    context_chars=context_chars,
    k=8, 
    sample_size=32,
    similarity_fn=name_similarity,
    output_path=output_path,
    resume_from=resume_from,
)

{
    "device": result.device.type,
    "training_items": len(result.dataset),
    "parameters": result.parameter_count,
    "final_loss": result.final_loss,
    "saved_to": str(output_path),
}


training tiny embedder:   0%|          | 0/50 [00:00<?, ?epoch/s]

KeyboardInterrupt: 

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, len(result.losses) + 1), result.losses, marker="o")
ax.set_xlabel("Epoch")
ax.set_ylabel("Triplet/ranking loss")
ax.set_title("Tiny string preselection training loss")
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()


In [ ]:
queries = ["apple", "market", "winter", "rocket"]
preview_count = 10

for query in queries:
    candidates = nearest(result.model, query, words, n=candidate_count)
    print(f"{query}: showing {preview_count} of {len(candidates)} candidates")
    for word, embedding_distance, edit_distance in candidates[:preview_count]:
        print(f"  {word:16s} embedding={embedding_distance:.3f} edit={edit_distance:.3f}")
    print()


## Dedupe preselection benchmark

This benchmark creates synthetic duplicate queries by applying small typo/edit mutations to held-out strings, then checks whether the original record appears in the model's top-k candidates.

How to read the numbers:

- `recall@k`: fraction of known duplicate queries where the true original appears within the first `k` candidates. For example, `recall@100 = 0.97` means 97% of duplicates would reach the downstream verifier if you keep 100 candidates.
- `median_rank`: the typical position of the true duplicate. Lower is better.
- `p95_rank`: 95% of duplicates rank at or above this position. For top-100 preselection, this should ideally be below 100.
- embedding/search timing: rough speed of this brute-force notebook path. Production should usually precompute candidate embeddings and use an ANN/vector index.

Rule of thumb for dedupe preselection:

- `recall@100 >= 0.99`: excellent.
- `0.95 <= recall@100 < 0.99`: promising, but validate on real labels.
- `0.90 <= recall@100 < 0.95`: risky unless the downstream process can tolerate misses.
- `< 0.90`: not good enough for a first-stage preselector.

Treat this as a smoke benchmark. The real answer requires representative labeled duplicate pairs from your own data.


In [ ]:
def mutate_string(value, rng):
    chars = list(value)
    if len(chars) <= 2:
        return value

    op = rng.choice(["delete", "swap", "replace", "insert"])
    idx = rng.randrange(len(chars))
    alphabet = "abcdefghijklmnopqrstuvwxyz"

    if op == "delete" and len(chars) > 3:
        del chars[idx]
    elif op == "swap" and len(chars) > 3:
        idx = min(idx, len(chars) - 2)
        chars[idx], chars[idx + 1] = chars[idx + 1], chars[idx]
    elif op == "replace":
        chars[idx] = rng.choice(alphabet)
    elif op == "insert":
        chars.insert(idx, rng.choice(alphabet))

    mutated = "".join(chars)
    return mutated if mutated != value else value[::-1]


def build_synthetic_duplicate_pairs(source_words, pair_count=500, seed=7):
    rng = random.Random(seed)
    usable = [word for word in source_words if 5 <= len(word) <= context_chars]
    originals = rng.sample(usable, min(pair_count, len(usable)))
    return [(mutate_string(original, rng), original) for original in originals]


benchmark_pairs = build_synthetic_duplicate_pairs(words, pair_count=500)
benchmark_pairs[:10], len(benchmark_pairs)


In [ ]:
def benchmark_preselection(model, candidate_words, pairs, ks=(1, 10, 50, 100), device=device):
    query_words = [query for query, _ in pairs]
    target_to_index = {word: idx for idx, word in enumerate(candidate_words)}
    valid = [(query, target) for query, target in pairs if target in target_to_index]
    query_words = [query for query, _ in valid]
    target_indices = np.asarray([target_to_index[target] for _, target in valid], dtype=np.int64)

    start = time.perf_counter()
    candidate_embeddings = model.embed_words(candidate_words, device=device, batch_size=4096)
    candidate_seconds = time.perf_counter() - start

    start = time.perf_counter()
    query_embeddings = model.embed_words(query_words, device=device, batch_size=4096)
    query_seconds = time.perf_counter() - start

    start = time.perf_counter()
    scores = query_embeddings @ candidate_embeddings.T
    order = np.argsort(-scores, axis=1)
    search_seconds = time.perf_counter() - start

    ranks = np.empty(len(valid), dtype=np.int64)
    for row, target_idx in enumerate(target_indices):
        ranks[row] = int(np.flatnonzero(order[row] == target_idx)[0]) + 1

    metrics = {f"recall@{k}": float(np.mean(ranks <= k)) for k in ks}
    metrics.update(
        {
            "pairs": len(valid),
            "candidates": len(candidate_words),
            "median_rank": float(np.median(ranks)),
            "mean_rank": float(np.mean(ranks)),
            "p95_rank": float(np.percentile(ranks, 95)),
            "candidate_embed_seconds": candidate_seconds,
            "query_embed_seconds": query_seconds,
            "search_seconds": search_seconds,
            "queries_per_second": len(valid) / max(query_seconds + search_seconds, 1e-9),
        }
    )
    return metrics, ranks, order


def preselection_verdict(recall_at_100):
    if recall_at_100 >= 0.99:
        return "excellent", "Top-100 recall is excellent for synthetic duplicates. Validate on real labeled duplicate pairs before shipping."
    if recall_at_100 >= 0.95:
        return "promising", "Top-100 recall is promising, but misses may still matter. Validate on real labels and inspect the misses."
    if recall_at_100 >= 0.90:
        return "risky", "Top-100 recall is risky for dedupe preselection. Consider better training data, more epochs, a larger candidate count, or a stronger model."
    return "not good enough", "Top-100 recall is too low for a first-stage preselector because missed duplicates never reach the verifier."


benchmark_metrics, benchmark_ranks, benchmark_order = benchmark_preselection(
    result.model,
    words,
    benchmark_pairs,
    ks=(1, 10, 50, 100),
)

print("Recall: chance that the true duplicate is kept in the candidate set")
for k in (1, 10, 50, 100):
    value = benchmark_metrics[f"recall@{k}"]
    print(f"  recall@{k:<3} {value:6.3f}  -> {value * 100:5.1f}% of synthetic duplicates found within top {k}")

print("Rank quality: where the true duplicate appears; lower is better")
print(f"  median_rank {benchmark_metrics['median_rank']:6.1f}  -> typical duplicate rank")
print(f"  mean_rank   {benchmark_metrics['mean_rank']:6.1f}  -> average duplicate rank")
print(f"  p95_rank    {benchmark_metrics['p95_rank']:6.1f}  -> 95% of duplicates rank this high or better")

print("Scale and timing for this brute-force notebook run")
print(f"  pairs                 {benchmark_metrics['pairs']}")
print(f"  candidates            {benchmark_metrics['candidates']}")
print(f"  candidate embedding   {benchmark_metrics['candidate_embed_seconds']:.3f}s")
print(f"  query embedding       {benchmark_metrics['query_embed_seconds']:.3f}s")
print(f"  brute-force search    {benchmark_metrics['search_seconds']:.3f}s")
print(f"  query throughput      {benchmark_metrics['queries_per_second']:,.1f} queries/sec")

label, explanation = preselection_verdict(benchmark_metrics["recall@100"])
print(f"Synthetic verdict: {label.upper()}")
print(explanation)


In [ ]:
misses = np.flatnonzero(benchmark_ranks > candidate_count)
print(f"misses above top-{candidate_count}: {len(misses)} / {len(benchmark_pairs)}")

for row in misses[:10]:
    query, target = benchmark_pairs[int(row)]
    nearest_indices = benchmark_order[int(row), :5]
    nearest_words = [words[int(idx)] for idx in nearest_indices]
    print(f"query={query!r} target={target!r} rank={int(benchmark_ranks[int(row)])} nearest={nearest_words}")


In [ ]:
assert result.parameter_count <= MODEL_PARAMETER_LIMIT
assert candidate_count == 100
assert output_path.exists()

print(f"Ready: {output_path} contains a {result.parameter_count:,}-parameter preselection model.")
